# 06 · Foundation models vs XGBoost across years

Re-run the comparison on the walk-forward folds from notebook 03.

**Key results**
- 2019 (148 origins, 56-day context): TimesFM 7.48 < Chronos 7.82 < XGBoost 8.48 < naive 8.77 MU.
- 2021 (96 origins, 28-day context): XGBoost 4.49 < TimesFM 4.80 < Chronos 5.28 < naive 5.35 MU.
- **The ranking flips between years.** Conclusion: a single validation year is not enough; the project needs a longer, locked external test and a cleaner, more recent data source.

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [07 · Dataset v2 and the 2024 distribution shift](07_v2_data_and_distribution_shift.ipynb)

Load data and lock the 2019 origins

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

DATA = Path("../data")

energy = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date"
)["energy_met_MU"].sort_index()

fold_a = pd.read_parquet(
    DATA / "processed/walkforward_fold_a.parquet"
)

origins = sorted(fold_a["origin"].unique())

valid_origins = []

for origin in origins:
    history = energy.loc[
        origin - pd.Timedelta(days=55):origin
    ]

    if len(history) == 56 and history.notna().all():
        valid_origins.append(origin)

assert len(valid_origins) == 148

evaluation = (
    fold_a[fold_a["origin"].isin(valid_origins)]
    .sort_values(["origin", "horizon"])
    .reset_index(drop=True)
)

assert len(evaluation) == 148 * 7

print("Evaluation origins:", len(valid_origins))
print("Prediction rows:", len(evaluation))

Evaluation origins: 148
Prediction rows: 1036


Load Chronos-Bolt Mini

In [2]:
from chronos import BaseChronosPipeline

chronos_model = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-mini",
    device_map="cpu",
    torch_dtype=torch.float32,
)

Forecast seven days

In [3]:
contexts = np.stack([
    energy.loc[
        origin - pd.Timedelta(days=55):origin
    ].to_numpy(dtype=np.float32)
    for origin in valid_origins
])

predictions = []

BATCH_SIZE = 32

for start in range(0, len(contexts), BATCH_SIZE):
    batch = torch.tensor(
        contexts[start:start+BATCH_SIZE]
    )

    with torch.inference_mode():
        quantiles, _ = chronos_model.predict_quantiles(
            inputs=batch,
            prediction_length=7,
            quantile_levels=[0.1, 0.5, 0.9],
        )

    predictions.append(
        quantiles[:, :, 1].cpu().numpy()
    )

predictions = np.concatenate(predictions)

assert predictions.shape == (148, 7)

evaluation["chronos_pred"] = predictions.reshape(-1)

actual = evaluation["target"].to_numpy()

chronos_mae = np.abs(
    actual - evaluation["chronos_pred"].to_numpy()
).mean()

xgb_mae = np.abs(
    actual - evaluation["xgb_pred"].to_numpy()
).mean()

naive_mae = np.abs(
    actual - evaluation["naive_pred"].to_numpy()
).mean()

print(f"Weekly naive MAE: {naive_mae:.3f} MU")
print(f"XGBoost MAE:      {xgb_mae:.3f} MU")
print(f"Chronos MAE:      {chronos_mae:.3f} MU")

Weekly naive MAE: 8.773 MU
XGBoost MAE:      8.478 MU
Chronos MAE:      7.820 MU


TimesFM on the same 148 origins

In [4]:
import timesfm as tfm
import numpy as np

timesfm_model = tfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

timesfm_model.compile(
    tfm.ForecastConfig(
        max_context=56,
        max_horizon=7,
        normalize_inputs=True,
        per_core_batch_size=8,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

BATCH_SIZE = 8
all_preds = []

for start in range(0, len(contexts), BATCH_SIZE):
    batch = contexts[start:start + BATCH_SIZE]

    forecast, _ = timesfm_model.forecast(
        horizon=7,
        inputs=list(batch),
    )

    all_preds.append(np.asarray(forecast))

timesfm_predictions = np.concatenate(all_preds, axis=0)

assert timesfm_predictions.shape == (148, 7)

evaluation["timesfm_pred"] = timesfm_predictions.reshape(-1)

# Compare all four models on identical rows.
actual = evaluation["target"].to_numpy()

for name, col in [
    ("Weekly naive", "naive_pred"),
    ("XGBoost", "xgb_pred"),
    ("Chronos", "chronos_pred"),
    ("TimesFM", "timesfm_pred"),
]:
    error = actual - evaluation[col].to_numpy()

    print(
        f"{name:14s} "
        f"MAE={np.abs(error).mean():.3f} MU  "
        f"RMSE={np.sqrt(np.mean(error**2)):.3f} MU"
    )

evaluation.to_parquet(
    DATA / "processed/foundation_walkforward_2019.parquet",
    index=False,
)

Weekly naive   MAE=8.773 MU  RMSE=11.367 MU
XGBoost        MAE=8.478 MU  RMSE=10.997 MU
Chronos        MAE=7.820 MU  RMSE=10.261 MU
TimesFM        MAE=7.475 MU  RMSE=10.099 MU


Diagnose why the ranking changes in 2019 data vs 2022

In [5]:
evaluation["month"] = evaluation["target_date"].dt.month

models = {
    "Weekly naive": "naive_pred",
    "XGBoost": "xgb_pred",
    "Chronos": "chronos_pred",
    "TimesFM": "timesfm_pred",
}

# Calculate absolute errors.
for name, col in models.items():
    evaluation[f"{name}_error"] = (
        evaluation["target"] - evaluation[col]
    ).abs()

error_cols = [f"{name}_error" for name in models]

print("\nMAE BY HORIZON")
display(
    evaluation.groupby("horizon")[error_cols]
    .mean()
    .round(3)
)

print("\nMAE BY MONTH")
display(
    evaluation.groupby("month")[error_cols]
    .agg(["mean", "count"])
    .round(3)
)


MAE BY HORIZON


,Weekly naive_error,XGBoost_error,Chronos_error,TimesFM_error
horizon,,,,
1,9.195,7.243,4.683,4.314
2,9.090,7.765,6.566,5.943
3,8.876,8.377,7.518,6.934
4,8.704,8.626,8.220,7.766
5,8.571,8.918,8.738,8.597
6,8.491,9.094,9.320,9.165
7,8.482,9.323,9.693,9.605



MAE BY MONTH


Weekly naive_error       XGBoost_error       Chronos_error        \
                    mean count          mean count          mean count   
month                                                                    
1                  2.835    49         4.350    49         3.129    49   
2                  2.718   147         1.656   147         2.067   147   
4                 21.379    28        10.138    28         7.261    28   
5                  8.684   217         6.609   217         9.382   217   
6                 17.943    42        11.882    42        13.760    42   
8                 10.752   161         8.399   161         9.797   161   
9                 10.263   210        13.272   210         9.839   210   
10                 8.048   126        11.335   126         6.502   126   
12                 7.379    56         9.677    56         6.501    56   

      TimesFM_error        
               mean count  
month                      
1             2.431    49  
2             1.771   147  
4             9.736    28  
5             8.348   217  
6            12.844    42  
8             9.887   161  
9             9.628   210  
10            6.354   126  
12            5.836    56

Lock the 2021 evaluation manifest

In [6]:
fold_b = pd.read_parquet(
    DATA / "processed/walkforward_fold_b.parquet"
)

evaluation_2021 = (
    fold_b
    .sort_values(["origin", "horizon"])
    .reset_index(drop=True)
)

origins_2021 = (
    evaluation_2021["origin"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

assert len(origins_2021) == 96
assert len(evaluation_2021) == 96 * 7

contexts_2021 = np.stack([
    energy.loc[
        origin - pd.Timedelta(days=27):origin
    ].to_numpy(dtype=np.float32)
    for origin in origins_2021
])

assert contexts_2021.shape == (96, 28)
assert np.isfinite(contexts_2021).all()

print("2021 evaluation origins:", len(origins_2021))
print("Forecast rows:", len(evaluation_2021))
print("Historical contexts:", contexts_2021.shape)

evaluation_2021.to_parquet(
    DATA / "processed/foundation_eval_2021_28d.parquet",
    index=False
)

2021 evaluation origins: 96
Forecast rows: 672
Historical contexts: (96, 28)


Load the models again

In [4]:
import torch
import timesfm as tfm
from chronos import BaseChronosPipeline

# Chronos-Bolt Mini
chronos_model = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-mini",
    device_map="cpu",
    torch_dtype=torch.float32,
)

# TimesFM 2.5
timesfm_model = tfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

timesfm_model.compile(
    tfm.ForecastConfig(
        max_context=28,
        max_horizon=7,
        normalize_inputs=True,
        per_core_batch_size=8,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("Chronos and TimesFM loaded successfully.")

Chronos and TimesFM loaded successfully.


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("../data")

energy = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date"
)["energy_met_MU"].sort_index()

evaluation_2021 = pd.read_parquet(
    DATA / "processed/foundation_eval_2021_28d.parquet"
).sort_values(["origin", "horizon"]).reset_index(drop=True)

origins_2021 = (
    evaluation_2021["origin"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

contexts_2021 = np.stack([
    energy.loc[
        origin - pd.Timedelta(days=27):origin
    ].to_numpy(dtype=np.float32)
    for origin in origins_2021
])

assert contexts_2021.shape == (96, 28)
assert np.isfinite(contexts_2021).all()
assert len(evaluation_2021) == 672

print("Contexts:", contexts_2021.shape)
print("Evaluation rows:", len(evaluation_2021))

Contexts: (96, 28)
Evaluation rows: 672


Chronos zero-shot (2021)

In [6]:
import torch
import numpy as np

CHRONOS_BATCH = 32
chronos_preds = []

for start in range(0, len(contexts_2021), CHRONOS_BATCH):
    batch = torch.tensor(
        contexts_2021[start:start + CHRONOS_BATCH],
        dtype=torch.float32
    )

    with torch.inference_mode():
        quantiles, _ = chronos_model.predict_quantiles(
            inputs=batch,
            prediction_length=7,
            quantile_levels=[0.1, 0.5, 0.9],
        )

    chronos_preds.append(
        quantiles[:, :, 1].cpu().numpy()
    )

chronos_preds = np.concatenate(chronos_preds, axis=0)

assert chronos_preds.shape == (96, 7)

evaluation_2021["chronos_pred"] = chronos_preds.reshape(-1)

print("Chronos predictions:", chronos_preds.shape)

Chronos predictions: (96, 7)


TimesFM zero-shot (2021)

In [7]:
import timesfm as tfm

timesfm_model.compile(
    tfm.ForecastConfig(
        max_context=28,
        max_horizon=7,
        normalize_inputs=True,
        per_core_batch_size=8,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

timesfm_preds = []

for start in range(0, len(contexts_2021), 8):
    batch = contexts_2021[start:start + 8]

    forecast, _ = timesfm_model.forecast(
        horizon=7,
        inputs=list(batch),
    )

    timesfm_preds.append(np.asarray(forecast))

timesfm_preds = np.concatenate(timesfm_preds, axis=0)

assert timesfm_preds.shape == (96, 7)

evaluation_2021["timesfm_pred"] = timesfm_preds.reshape(-1)

print("TimesFM predictions:", timesfm_preds.shape)

TimesFM predictions: (96, 7)


Compare all four models

In [8]:
actual = evaluation_2021["target"].to_numpy()

model_columns = {
    "Weekly naive": "naive_pred",
    "XGBoost": "xgb_pred",
    "Chronos": "chronos_pred",
    "TimesFM": "timesfm_pred",
}

results_2021 = []

for name, col in model_columns.items():
    error = actual - evaluation_2021[col].to_numpy()

    results_2021.append({
        "Model": name,
        "MAE": np.abs(error).mean(),
        "RMSE": np.sqrt(np.mean(error**2)),
        "Bias": error.mean(),
    })

results_2021 = (
    pd.DataFrame(results_2021)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(results_2021.round(3))

evaluation_2021.to_parquet(
    DATA / "processed/foundation_walkforward_2021.parquet",
    index=False
)

,Model,MAE,RMSE,Bias
0,XGBoost,4.490,7.081,1.251
1,TimesFM,4.799,7.685,0.999
2,Chronos,5.276,7.762,2.071
3,Weekly naive,5.353,8.294,1.063
